In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import gcamreader
import pandas as pd

dfDef = pd.read_csv("../resources/gdpdef.csv")
dfDef.head()
dfDef['Year'] = dfDef['date'].str.split('-').str[0].astype(int)
arrDef = dfDef.set_index('Year')['gdpdef']
arrDef[2024] = (139.71495 / 136.41462) * arrDef[2023]
def gcam_deflator(value, from_year=2023, to_year=1975):
    return value * arrDef[to_year] / arrDef[from_year]

dfExc = pd.read_csv("../resources/DEXKOUS.csv")
dfExc['date'] = pd.to_datetime(dfExc['observation_date'])
dfExcAnnual = dfExc.groupby(dfExc['date'].dt.year)['DEXKOUS'].mean()

def string_to_xml_file(xml_string, file_name):
    """
    Converts a string into a well-formatted (indented) XML file, without unnecessary newlines.

    Parameters:
    xml_string (str): The XML content as a string.
    file_name (str): The desired filename for the XML file.
    """
    try:
        # Parse the XML string
        root = ET.ElementTree(ET.fromstring(xml_string))
        
        # Convert ElementTree to a string
        rough_string = ET.tostring(root.getroot(), encoding="utf-8")
        
        # Use minidom to pretty-print the XML
        parsed = minidom.parseString(rough_string)
        pretty_xml_as_string = parsed.toprettyxml(indent="  ")
        
        # Remove unnecessary blank lines created by toprettyxml()
        pretty_xml_as_string = "\n".join([line for line in pretty_xml_as_string.splitlines() if line.strip()])
        
        # Write the formatted XML to a file
        with open(file_name, "w", encoding="utf-8") as f:
            f.write(pretty_xml_as_string)
        
        print(f"XML file '{file_name}' created successfully with proper indentation and no extra newlines.")
    except ET.ParseError as e:
        print("Error parsing XML string:", e)

# Source

## Data Sources
- Ministry of Trade, Industry and Energy (MOTIE). New and Renewable Energy Deployment Support Program.
- Collected Data Repo: `../resources/RESP/`

## Implemented Input Files
- `/input/policy/korea-2035/buildings/h2_subsidy_comm_cp.xml`  
- `/input/policy/korea-2035/buildings/h2_subsidy_comm_ep.xml`
- `/input/policy/korea-2035/buildings/h2_subsidy_resid_cp.xml`  
- `/input/policy/korea-2035/buildings/h2_subsidy_resid_cp.xml`  
- `/input/policy/korea-2035/buildings/pv_subsidy_comm_cp.xml`  
- `/input/policy/korea-2035/buildings/pv_subsidy_comm_ep.xml`
- `/input/policy/korea-2035/buildings/pv_subsidy_resid_cp.xml`  
- `/input/policy/korea-2035/buildings/pv_subsidy_resid_cp.xml`  

# Renewable Energy Subsidy for Buildings

We collected data for the past three years (2021–2023) from the *New and Renewable Energy Deployment Support Program for the Building Sector* (including commercial and residential buildings) administered by the Ministry of Trade, Industry and Energy (MOTIE). 
Collected document directory: `../resources/RESP/`.
For energy supply using building-integrated solar PV (rooftop PV) and hydrogen, we modeled subsidies based on the average values over this period.  

In the *Enhanced Ambition* scenario, we assume that starting in 2030, subsidies are provided at the highest annual level observed in the past, corresponding to the 2017 subsidy level.

Detailed implementations are presented below.

# Implementaion of Subsidy for Commercial Buildings

## Rooftop PV

In [7]:
fuelcell_subsidy_2021 = 18200 / dfExcAnnual[2021] * gcam_deflator(1, 2021, 1975)
fuelcell_subsidy_2022 = 18200 / dfExcAnnual[2022] * gcam_deflator(1, 2022, 1975)
fuelcell_subsidy_2023 = 7860 / dfExcAnnual[2023] * gcam_deflator(1, 2023, 1975)
fuelcell_subsidy_cp = (fuelcell_subsidy_2021 + fuelcell_subsidy_2022 + fuelcell_subsidy_2023) / 3
fuelcell_subsidy_ep = 22000 / dfExcAnnual[2017] * gcam_deflator(1, 2017, 1975)
fuelcell_subsidy_cp, fuelcell_subsidy_ep

(np.float64(2.8991205867312364), np.float64(5.416974470056469))

In [8]:
kw_to_mwh = 8.760
mwh_to_gj = 3.6
kw_to_gj = kw_to_mwh * mwh_to_gj
gj_to_kw = 1 / kw_to_gj

In [9]:
fuelcell_subsidy_1975USD_GJ_cp = fuelcell_subsidy_cp * gj_to_kw
fuelcell_subsidy_1975USD_GJ_ep = fuelcell_subsidy_ep * gj_to_kw
fuelcell_subsidy_1975USD_GJ_cp, fuelcell_subsidy_1975USD_GJ_ep

(np.float64(0.09193051074109704), np.float64(0.17177113362685403))

In [10]:
pv_subsidy_2023KRW = 972 # (1181 * 2 + 936 * 1) / 3
pv_subsidy_1975USD = pv_subsidy_2023KRW / dfExcAnnual[2023] * gcam_deflator(1, 2023, 1975)
pv_subsidy_1975USD

np.float64(0.16911550214394339)

In [11]:
pv_capacity_factor = 0.177842888977991 # imported rooftop_pv technology capacity factor in South Korea region from electricity_water.xml
kw_to_mwh = 8.760 * pv_capacity_factor
mwh_to_gj = 3.6
kw_to_gj = kw_to_mwh * mwh_to_gj
gj_to_kw = 1 / kw_to_gj

In [12]:
pv_subsidy_1975USD_GJ = pv_subsidy_1975USD * gj_to_kw
pv_subsidy_1975USD_GJ

np.float64(0.030153679042393373)

In [13]:
proj_path = Path("../../")
xml_path = proj_path / "input" / "gcamdata" / "xml"
db_path = proj_path / "output"

In [14]:
xml_file_path = xml_path / "building_det.xml"
tree = ET.parse(xml_file_path)
root = tree.getroot()  # Get the root element of the XML
korea = root.find(".//region[@name='South Korea']")

In [15]:
# Create the new root for the reproduced XML
new_root = ET.Element("scenario")
new_world = ET.SubElement(new_root, "world")
new_korea = ET.SubElement(new_world, "region", {'name': "South Korea"})
for supplysector in korea.findall(".//supplysector"):
    supplysectorNm = supplysector.get('name')
    if "comm" not in supplysectorNm:
        continue
    # print(supplysectorNm)
    new_supplysector = ET.Element('supplysector', {'name': supplysectorNm})
    
    for subsector in supplysector.findall(".//subsector"):
        new_subsector = ET.Element('subsector', {'name': subsector.get('name')})

        for stub_technology in subsector.findall(".//stub-technology"):
            stub_technology_nm = stub_technology.get("name")
            if stub_technology_nm != 'electricity':
                continue
            new_stub_technology = ET.Element('stub-technology', {'name': 'electricity'})
            for year in range(2020, 2030, 5):
                period = ET.SubElement(new_stub_technology, 'period', {'year': str(year)})
                minicam_non_energy_input = ET.SubElement(period, 'minicam-non-energy-input', {'name': "renewable-energy-subsidy-resid"})
                input_cost = ET.SubElement(minicam_non_energy_input, 'input-cost')
                input_cost.text = f"{-pv_subsidy_1975USD_GJ:.3f}"#str(-pv_subsidy_1975USD_GJ)
            
            if new_stub_technology:
                new_subsector.append(new_stub_technology)
        
        if new_subsector:
            new_supplysector.append(new_subsector)
    if new_supplysector:
        new_korea.append(new_supplysector)

In [ ]:
outfile_path = proj_path / "input" / "policy" / "korea-2035" / "buildings" / "pv_subsidy_comm_cp.xml"

# save
xml_string = ET.tostring(new_root, encoding="unicode")
string_to_xml_file(xml_string, outfile_path)

XML file '/data/project/tae/gcam-core/input/policy/ndc/buildings/pv_subsidy_comm_cp.xml' created successfully with proper indentation and no extra newlines.


### Enhanced Ambition

In [123]:
# Create the new root for the reproduced XML
new_root = ET.Element("scenario")
new_world = ET.SubElement(new_root, "world")
new_korea = ET.SubElement(new_world, "region", {'name': "South Korea"})
for supplysector in korea.findall(".//supplysector"):
    supplysectorNm = supplysector.get('name')
    if "comm" not in supplysectorNm:
        continue
    # print(supplysectorNm)
    new_supplysector = ET.Element('supplysector', {'name': supplysectorNm})
    
    for subsector in supplysector.findall(".//subsector"):
        new_subsector = ET.Element('subsector', {'name': subsector.get('name')})

        for stub_technology in subsector.findall(".//stub-technology"):
            stub_technology_nm = stub_technology.get("name")
            if stub_technology_nm != 'electricity':
                continue
            new_stub_technology = ET.Element('stub-technology', {'name': 'electricity'})
            for year in range(2020, 2040, 5):
                period = ET.SubElement(new_stub_technology, 'period', {'year': str(year)})
                minicam_non_energy_input = ET.SubElement(period, 'minicam-non-energy-input', {'name': "renewable-energy-subsidy-resid"})
                input_cost = ET.SubElement(minicam_non_energy_input, 'input-cost')
                input_cost.text = f"{-pv_subsidy_1975USD_GJ:.3f}"
            
            if new_stub_technology:
                new_subsector.append(new_stub_technology)
        
        if new_subsector:
            new_supplysector.append(new_subsector)
    if new_supplysector:
        new_korea.append(new_supplysector)

In [ ]:
outfile_path = proj_path / "input" / "policy" / "korea-2035" / "buildings" / "pv_subsidy_comm_ep.xml"

# save
xml_string = ET.tostring(new_root, encoding="unicode")
string_to_xml_file(xml_string, outfile_path)

XML file '/data/project/tae/gcam-core/input/policy/ndc/buildings/pv_subsidy_comm_ep.xml' created successfully with proper indentation and no extra newlines.


## Hydrogen

In [126]:
proj_path = Path("/data/project/tae/gcam-core")
xml_path = proj_path / "input" / "gcamdata" / "xml"
db_path = proj_path / "output"

In [127]:
bld_file_path = xml_path / "building_det.xml"
tree = ET.parse(bld_file_path)
root = tree.getroot()  # Get the root element of the XML
korea = root.find(".//region[@name='South Korea']")

In [128]:
visualize_xml_tree(korea, max_depth=3)

- region (Attributes: {'name': 'South Korea'})
  - supplysector (Attributes: {'name': 'resid heating modern_d1'})
    - keyword (Attributes: {'final-energy': 'building'})
    - relative-cost-logit
      - logit-exponent (Attributes: {'fillout': '1', 'year': '1975'})
    - output-unit (Text: EJ)
    - input-unit (Text: EJ)
    - price-unit (Text: 1975$/GJ)
    - subsector (Attributes: {'name': 'biomass'})
      - relative-cost-logit
      - fuelprefElasticity (Attributes: {'fillout': '1', 'year': '1975'})
      - stub-technology (Attributes: {'name': 'biomass'})
      - share-weight (Attributes: {'year': '1975'})
      - share-weight (Attributes: {'year': '1990'})
      - share-weight (Attributes: {'year': '2005'})
      - share-weight (Attributes: {'year': '2010'})
      - share-weight (Attributes: {'year': '2015'})
      - share-weight (Attributes: {'fillout': '1', 'year': '1975'})
      - interpolation-rule (Attributes: {'apply-to': 'share-weight', 'from-year': '2015', 'to-year': '21

In [129]:
# Create the new root for the reproduced XML
new_root = ET.Element("scenario")
new_world = ET.SubElement(new_root, "world")
new_korea = ET.SubElement(new_world, "region", {'name': "South Korea"})
for supplysector in korea.findall(".//supplysector"):
    if "comm" not in supplysector.get('name'):
        continue
    new_supplysector = ET.Element('supplysector', {'name': supplysector.get('name')})

    for subsector in supplysector.findall(".//subsector"):
        new_subsector = ET.Element('subsector', {'name': subsector.get('name')})

        for stub_technology in subsector.findall(".//stub-technology"):
            stub_technology_nm = stub_technology.get("name")
            if stub_technology_nm != 'hydrogen':
                continue
            new_stub_technology = ET.Element('stub-technology', {'name': stub_technology_nm})
            for year in range(2020, 2040, 5):
                period = ET.SubElement(new_stub_technology, 'period', {'year': str(year)})
                minicam_non_energy_input = ET.SubElement(period, 'minicam-non-energy-input', {'name': "renewable-energy-subsidy"})
                input_cost = ET.SubElement(minicam_non_energy_input, 'input-cost')
                input_cost.text = f"{-fuelcell_subsidy_1975USD_GJ_cp:.3f}"#str(-h2_subsidy_1975USD_GJ)
            
            if new_stub_technology:
                new_subsector.append(new_stub_technology)
        
        if new_subsector:
            new_supplysector.append(new_subsector)
    if new_supplysector:
        new_korea.append(new_supplysector)

In [130]:
visualize_xml_tree(new_root, max_depth=8)

- scenario
  - world
    - region (Attributes: {'name': 'South Korea'})
      - supplysector (Attributes: {'name': 'comm heating'})
        - subsector (Attributes: {'name': 'gas'})
          - stub-technology (Attributes: {'name': 'hydrogen'})
            - period (Attributes: {'year': '2020'})
              - minicam-non-energy-input (Attributes: {'name': 'renewable-energy-subsidy'})
                - input-cost (Text: -0.092)
            - period (Attributes: {'year': '2025'})
              - minicam-non-energy-input (Attributes: {'name': 'renewable-energy-subsidy'})
                - input-cost (Text: -0.092)
            - period (Attributes: {'year': '2030'})
              - minicam-non-energy-input (Attributes: {'name': 'renewable-energy-subsidy'})
                - input-cost (Text: -0.092)
            - period (Attributes: {'year': '2035'})
              - minicam-non-energy-input (Attributes: {'name': 'renewable-energy-subsidy'})
                - input-cost (Text: -0.092)
   

In [ ]:
outfile_path = proj_path / "input" / "policy" / "korea-2035" / "buildings" / "h2_subsidy_comm_cp.xml"

# save
xml_string = ET.tostring(new_root, encoding="unicode")
string_to_xml_file(xml_string, outfile_path)

XML file '/data/project/tae/gcam-core/input/policy/ndc/buildings/h2_subsidy_comm_cp.xml' created successfully with proper indentation and no extra newlines.


### Enhanced Ambition

In [132]:
# Create the new root for the reproduced XML
new_root = ET.Element("scenario")
new_world = ET.SubElement(new_root, "world")
new_korea = ET.SubElement(new_world, "region", {'name': "South Korea"})
for supplysector in korea.findall(".//supplysector"):
    if "comm" not in supplysector.get('name'):
        continue
    new_supplysector = ET.Element('supplysector', {'name': supplysector.get('name')})

    for subsector in supplysector.findall(".//subsector"):
        new_subsector = ET.Element('subsector', {'name': subsector.get('name')})

        for stub_technology in subsector.findall(".//stub-technology"):
            stub_technology_nm = stub_technology.get("name")
            if stub_technology_nm != 'hydrogen':
                continue
            new_stub_technology = ET.Element('stub-technology', {'name': stub_technology_nm})
            for year in range(2020, 2030, 5):
                period = ET.SubElement(new_stub_technology, 'period', {'year': str(year)})
                minicam_non_energy_input = ET.SubElement(period, 'minicam-non-energy-input', {'name': "renewable-energy-subsidy"})
                input_cost = ET.SubElement(minicam_non_energy_input, 'input-cost')
                input_cost.text = f"{-fuelcell_subsidy_1975USD_GJ_cp:.3f}"#str(-h2_subsidy_1975USD_GJ)
            for year in range(2030, 2040, 5):
                period = ET.SubElement(new_stub_technology, 'period', {'year': str(year)})
                minicam_non_energy_input = ET.SubElement(period, 'minicam-non-energy-input', {'name': "renewable-energy-subsidy"})
                input_cost = ET.SubElement(minicam_non_energy_input, 'input-cost')
                input_cost.text = f"{-fuelcell_subsidy_1975USD_GJ_ep:.3f}"#str(-h2_subsidy_1975USD_GJ)
            
            if new_stub_technology:
                new_subsector.append(new_stub_technology)
        
        if new_subsector:
            new_supplysector.append(new_subsector)
    if new_supplysector:
        new_korea.append(new_supplysector)

In [133]:
visualize_xml_tree(new_root, max_depth=9)

- scenario
  - world
    - region (Attributes: {'name': 'South Korea'})
      - supplysector (Attributes: {'name': 'comm heating'})
        - subsector (Attributes: {'name': 'gas'})
          - stub-technology (Attributes: {'name': 'hydrogen'})
            - period (Attributes: {'year': '2020'})
              - minicam-non-energy-input (Attributes: {'name': 'renewable-energy-subsidy'})
                - input-cost (Text: -0.092)
            - period (Attributes: {'year': '2025'})
              - minicam-non-energy-input (Attributes: {'name': 'renewable-energy-subsidy'})
                - input-cost (Text: -0.092)
            - period (Attributes: {'year': '2030'})
              - minicam-non-energy-input (Attributes: {'name': 'renewable-energy-subsidy'})
                - input-cost (Text: -0.172)
            - period (Attributes: {'year': '2035'})
              - minicam-non-energy-input (Attributes: {'name': 'renewable-energy-subsidy'})
                - input-cost (Text: -0.172)
   

In [ ]:
outfile_path = proj_path / "input" / "policy" / "korea-2035" / "buildings" / "h2_subsidy_comm_ep.xml"

# save
xml_string = ET.tostring(new_root, encoding="unicode")
string_to_xml_file(xml_string, outfile_path)

XML file '/data/project/tae/gcam-core/input/policy/ndc/buildings/h2_subsidy_comm_ep.xml' created successfully with proper indentation and no extra newlines.


# Implementaion of Subsidy for Residential Buildings

## Rooftop PV

In [16]:
pv_subsidy_2023KRW = 1099.33 # (1181 * 2 + 936 * 1) / 3
pv_subsidy_1975USD = pv_subsidy_2023KRW / dfExcAnnual[2023] * gcam_deflator(1, 2023, 1975)
pv_subsidy_1975USD

np.float64(0.19126928495051568)

In [17]:
pv_capacity_factor = 0.177842888977991 # imported rooftop_pv technology capacity factor in South Korea region from electricity_water.xml
kw_to_mwh = 8.760 * pv_capacity_factor
mwh_to_gj = 3.6
kw_to_gj = kw_to_mwh * mwh_to_gj
gj_to_kw = 1 / kw_to_gj

In [19]:
pv_subsidy_1975USD_GJ = pv_subsidy_1975USD * gj_to_kw
pv_subsidy_1975USD_GJ

np.float64(0.034103748952339814)

In [20]:
xml_file_path = xml_path / "building_det.xml"
tree = ET.parse(xml_file_path)
root = tree.getroot()  # Get the root element of the XML
korea = root.find(".//region[@name='South Korea']")

In [21]:
# Create the new root for the reproduced XML
new_root = ET.Element("scenario")
new_world = ET.SubElement(new_root, "world")
new_korea = ET.SubElement(new_world, "region", {'name': "South Korea"})
for supplysector in korea.findall(".//supplysector"):
    supplysectorNm = supplysector.get('name')
    if "resid" not in supplysectorNm:
        continue
    # print(supplysectorNm)
    new_supplysector = ET.Element('supplysector', {'name': supplysectorNm})
    
    for subsector in supplysector.findall(".//subsector"):
        new_subsector = ET.Element('subsector', {'name': subsector.get('name')})

        for stub_technology in subsector.findall(".//stub-technology"):
            stub_technology_nm = stub_technology.get("name")
            if stub_technology_nm != 'electricity':
                continue
            new_stub_technology = ET.Element('stub-technology', {'name': 'electricity'})
            for year in range(2020, 2030, 5):
                period = ET.SubElement(new_stub_technology, 'period', {'year': str(year)})
                minicam_non_energy_input = ET.SubElement(period, 'minicam-non-energy-input', {'name': "renewable-energy-subsidy-resid"})
                input_cost = ET.SubElement(minicam_non_energy_input, 'input-cost')
                input_cost.text = f"{-pv_subsidy_1975USD_GJ:.3f}"#str(-pv_subsidy_1975USD_GJ)
            
            if new_stub_technology:
                new_subsector.append(new_stub_technology)
        
        if new_subsector:
            new_supplysector.append(new_subsector)
    if new_supplysector:
        new_korea.append(new_supplysector)

In [ ]:
outfile_path = proj_path / "input" / "policy" / "korea-2035" / "buildings" / "pv_subsidy_resid_cp.xml"

# save
xml_string = ET.tostring(new_root, encoding="unicode")
string_to_xml_file(xml_string, outfile_path)

XML file '/data/project/tae/gcam-core/input/policy/ndc/buildings/pv_subsidy_resid_cp.xml' created successfully with proper indentation and no extra newlines.


### Enhanced Ambition

In [22]:
# Create the new root for the reproduced XML
new_root = ET.Element("scenario")
new_world = ET.SubElement(new_root, "world")
new_korea = ET.SubElement(new_world, "region", {'name': "South Korea"})
for supplysector in korea.findall(".//supplysector"):
    supplysectorNm = supplysector.get('name')
    if "resid" not in supplysectorNm:
        continue
    # print(supplysectorNm)
    new_supplysector = ET.Element('supplysector', {'name': supplysectorNm})
    
    for subsector in supplysector.findall(".//subsector"):
        new_subsector = ET.Element('subsector', {'name': subsector.get('name')})

        for stub_technology in subsector.findall(".//stub-technology"):
            stub_technology_nm = stub_technology.get("name")
            if stub_technology_nm != 'electricity':
                continue
            new_stub_technology = ET.Element('stub-technology', {'name': 'electricity'})
            for year in range(2020, 2040, 5):
                period = ET.SubElement(new_stub_technology, 'period', {'year': str(year)})
                minicam_non_energy_input = ET.SubElement(period, 'minicam-non-energy-input', {'name': "renewable-energy-subsidy-resid"})
                input_cost = ET.SubElement(minicam_non_energy_input, 'input-cost')
                input_cost.text = f"{-pv_subsidy_1975USD_GJ:.3f}"
            
            if new_stub_technology:
                new_subsector.append(new_stub_technology)
        
        if new_subsector:
            new_supplysector.append(new_subsector)
    if new_supplysector:
        new_korea.append(new_supplysector)

In [ ]:
outfile_path = proj_path / "input" / "policy" / "korea-2035" / "buildings" / "pv_subsidy_resid_ep.xml"

# save
xml_string = ET.tostring(new_root, encoding="unicode")
string_to_xml_file(xml_string, outfile_path)

XML file '/data/project/tae/gcam-core/input/policy/ndc/buildings/pv_subsidy_resid_ep.xml' created successfully with proper indentation and no extra newlines.


## Hydrogen

In [ ]:
bld_file_path = xml_path / "building_det.xml"
tree = ET.parse(bld_file_path)
root = tree.getroot()  # Get the root element of the XML
korea = root.find(".//region[@name='South Korea']")

In [ ]:
# Create the new root for the reproduced XML
new_root = ET.Element("scenario")
new_world = ET.SubElement(new_root, "world")
new_korea = ET.SubElement(new_world, "region", {'name': "South Korea"})
for supplysector in korea.findall(".//supplysector"):
    if "resid" not in supplysector.get('name'):
        continue
    new_supplysector = ET.Element('supplysector', {'name': supplysector.get('name')})

    for subsector in supplysector.findall(".//subsector"):
        new_subsector = ET.Element('subsector', {'name': subsector.get('name')})

        for stub_technology in subsector.findall(".//stub-technology"):
            stub_technology_nm = stub_technology.get("name")
            if stub_technology_nm != 'hydrogen':
                continue
            new_stub_technology = ET.Element('stub-technology', {'name': stub_technology_nm})
            for year in range(2020, 2040, 5):
                period = ET.SubElement(new_stub_technology, 'period', {'year': str(year)})
                minicam_non_energy_input = ET.SubElement(period, 'minicam-non-energy-input', {'name': "renewable-energy-subsidy"})
                input_cost = ET.SubElement(minicam_non_energy_input, 'input-cost')
                input_cost.text = f"{-fuelcell_subsidy_1975USD_GJ_cp:.3f}"#str(-h2_subsidy_1975USD_GJ)
            
            if new_stub_technology:
                new_subsector.append(new_stub_technology)
        
        if new_subsector:
            new_supplysector.append(new_subsector)
    if new_supplysector:
        new_korea.append(new_supplysector)

In [ ]:
outfile_path = proj_path / "input" / "policy" / "korea-2035" / "buildings" / "h2_subsidy_resid_cp.xml"

# save
xml_string = ET.tostring(new_root, encoding="unicode")
string_to_xml_file(xml_string, outfile_path)

XML file '/data/project/tae/gcam-core/input/policy/ndc/buildings/h2_subsidy_resid_cp.xml' created successfully with proper indentation and no extra newlines.


### Enhanced Ambition

In [ ]:
# Create the new root for the reproduced XML
new_root = ET.Element("scenario")
new_world = ET.SubElement(new_root, "world")
new_korea = ET.SubElement(new_world, "region", {'name': "South Korea"})
for supplysector in korea.findall(".//supplysector"):
    if 'resid' not in supplysector.get('name'):
        continue
    new_supplysector = ET.Element('supplysector', {'name': supplysector.get('name')})

    for subsector in supplysector.findall(".//subsector"):
        new_subsector = ET.Element('subsector', {'name': subsector.get('name')})

        for stub_technology in subsector.findall(".//stub-technology"):
            stub_technology_nm = stub_technology.get("name")
            if stub_technology_nm != 'hydrogen':
                continue
            new_stub_technology = ET.Element('stub-technology', {'name': stub_technology_nm})
            for year in range(2020, 2030, 5):
                period = ET.SubElement(new_stub_technology, 'period', {'year': str(year)})
                minicam_non_energy_input = ET.SubElement(period, 'minicam-non-energy-input', {'name': "renewable-energy-subsidy"})
                input_cost = ET.SubElement(minicam_non_energy_input, 'input-cost')
                input_cost.text = f"{-fuelcell_subsidy_1975USD_GJ_cp:0.3}"#str(-h2_subsidy_1975USD_GJ)
            for year in range(2030, 2040, 5):
                period = ET.SubElement(new_stub_technology, 'period', {'year': str(year)})
                minicam_non_energy_input = ET.SubElement(period, 'minicam-non-energy-input', {'name': "renewable-energy-subsidy"})
                input_cost = ET.SubElement(minicam_non_energy_input, 'input-cost')
                input_cost.text = f"{-fuelcell_subsidy_1975USD_GJ_ep:0.3}"#str(-h2_subsidy_1975USD_GJ)
            
            if new_stub_technology:
                new_subsector.append(new_stub_technology)
        
        if new_subsector:
            new_supplysector.append(new_subsector)
    if new_supplysector:
        new_korea.append(new_supplysector)

In [ ]:
visualize_xml_tree(new_root, max_depth=9)

- scenario
  - world
    - region (Attributes: {'name': 'South Korea'})
      - supplysector (Attributes: {'name': 'resid heating modern_d1'})
        - subsector (Attributes: {'name': 'gas'})
          - stub-technology (Attributes: {'name': 'hydrogen'})
            - period (Attributes: {'year': '2020'})
              - minicam-non-energy-input (Attributes: {'name': 'renewable-energy-subsidy'})
                - input-cost (Text: -0.0969)
            - period (Attributes: {'year': '2025'})
              - minicam-non-energy-input (Attributes: {'name': 'renewable-energy-subsidy'})
                - input-cost (Text: -0.0969)
            - period (Attributes: {'year': '2030'})
              - minicam-non-energy-input (Attributes: {'name': 'renewable-energy-subsidy'})
                - input-cost (Text: -0.183)
            - period (Attributes: {'year': '2035'})
              - minicam-non-energy-input (Attributes: {'name': 'renewable-energy-subsidy'})
                - input-cost (Text

In [ ]:
outfile_path = proj_path / "input" / "policy" / "korea-2035" / "buildings" / "h2_subsidy_resid_ep.xml"

# save
xml_string = ET.tostring(new_root, encoding="unicode")
string_to_xml_file(xml_string, outfile_path)

XML file '/data/project/tae/gcam-core/input/policy/ndc/buildings/h2_subsidy_resid_ep.xml' created successfully with proper indentation and no extra newlines.
